Replay mode uses recorded fixtures and does not call a live model. Set `NORTHSTAR_MODE=live` with `GEMINI_API_KEY` to opt in, or use `NORTHSTAR_MODE=record` to save synthetic responses.

In [ ]:
from pathlib import Path
from northstar.runtime import get_client, PromptRequest, Message
from lab01 import run_lab

client = get_client(Path("fixtures/replays.json"))  # NORTHSTAR_MODE=replay|live|record

# 01 — LLM Behavior and Prompt Anatomy

## Scenario

Northstar’s classifier changes behavior after a request configuration change. This lab tests one variable at a time against recorded responses.

## Baseline hypothesis

A precise instruction and approved evidence should classify clear cases while escalating the ambiguous payment case.

In [ ]:
results = run_lab(client)
assert results["baseline_accuracy"].numerator == 4
assert results["baseline_accuracy"].denominator == 4

## Experiment 1 — position is a variable

Moving evidence into padded middle context produces one wrong case in the recorded run.

In [ ]:
assert results["middle_accuracy"].numerator == 3
assert results["middle_accuracy"].denominator == 4

## Experiment 2 — sampling is a trade-off

The temperature 0 and 0.9 replay outputs differ for one ambiguous case.

In [ ]:
assert results["temperature_comparison"] == ("unknown", "refund")
print("Estimated padding tokens:", results["padding_tokens"].numerator)

## Failure injection — missing evidence

Weak instructions produce recorded `refund`; explicit abstention produces recorded `unknown`.

In [ ]:
assert results["weak_missing_evidence"].numerator == 1
assert results["abstention_missing_evidence"].numerator == 1

## Takeaway

Roles, evidence position, sampling, and fallback instructions are observable parts of a request contract.

## References

- [Core concepts and workflow](README.md#core-concepts--workflow)
- [Production best practices](README.md#production-best-practices)